# 🏆 Master Stacking Ensemble

**Goal:** Combine the power of ARIMA, ETS, LSTM, and GRU using a Stacking Meta-Learner to achieve the ultimate forecasting accuracy for personal expenses.

## ✅ Step 1: Environment Setup & Model Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import tensorflow as tf
from statsmodels.tsa.arima.model import ARIMAResults
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("✅ Environment ready.")

✅ Environment ready.


In [2]:
# Load preprocessed data
test_df = pd.read_csv('data/processed/test_monthly.csv', parse_dates=['date'], index_col='date')
test_df.index = pd.DatetimeIndex(test_df.index).to_period('M').to_timestamp('M')
test_actuals = test_df['amount']

print(f"📅 Test data loaded: {len(test_actuals)} months.")

📅 Test data loaded: 6 months.


In [3]:
# 1. Load statsmodels
arima_res = ARIMAResults.load('models/arima_hybrid_base.pkl')
# Note: ETS in statsmodels doesn't always support easy .load() for certain versions, 
# but we have the saved results/parameters or can recreate if needed. 
# Here we'll check for pkl.
with open('models/ets_hybrid_base.pkl', 'rb') as f:
    ets_res = pickle.load(f)

# 2. Load Deep Learning models
lstm_model = tf.keras.models.load_model('models/lstm_arima_residuals.h5', compile=False)
gru_model = tf.keras.models.load_model('models/gru_ets_residuals.h5', compile=False)

# 3. Load Scalers
with open('models/scaler_arima.pkl', 'rb') as f:
    scaler_arima = pickle.load(f)
with open('models/scaler_ets.pkl', 'rb') as f:
    scaler_ets = pickle.load(f)

print("✅ All models and scalers loaded successfully.")

TypeError: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}

## ✅ Step 2: Prediction Generation (Test Period)

We will generate forecasts from each of the 4 models for the test period. These will serve as the *features* for our Meta-Learner.

In [ ]:
steps = len(test_actuals)

# 1. ARIMA Base Forecast
arima_preds = arima_res.forecast(steps=steps).values

# 2. ETS Base Forecast
ets_preds = ets_res.forecast(steps=steps).values

# 3. Deep Learning Residual Forecasts
def get_nn_forecast(model, scaler, base_residuals_scaled, n_steps=3, steps=6):
    history = list(base_residuals_scaled[-n_steps:].flatten())
    preds = []
    for _ in range(steps):
        x_input = np.array(history[-n_steps:]).reshape((1, n_steps, 1))
        yhat = model.predict(x_input, verbose=0)[0][0]
        preds.append(yhat)
        history.append(yhat)
    return scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()

train_df = pd.read_csv('data/processed/train_monthly.csv', parse_dates=['date'], index_col='date')
train_df.index = pd.DatetimeIndex(train_df.index).to_period('M').to_timestamp('M')
train_amount = train_df['amount']

arima_train_res = train_amount - arima_res.fittedvalues
ets_train_res = train_amount - ets_res.fittedvalues

arima_res_scaled = scaler_arima.transform(arima_train_res.values.reshape(-1,1))
ets_res_scaled = scaler_ets.transform(ets_train_res.values.reshape(-1,1))

lstm_res_preds = get_nn_forecast(lstm_model, scaler_arima, arima_res_scaled)
gru_res_preds = get_nn_forecast(gru_model, scaler_ets, ets_res_scaled)

print("✅ Predictions generated for all 4 components.")

## ✅ Step 3: Joint Stacking (The Meta-Learner)

We construct a matrix containing all 4 signals and find the optimal way to combine them.

In [ ]:
# Create feature matrix for stacking
X_meta = np.column_stack([
    arima_preds, 
    ets_preds, 
    lstm_res_preds, 
    gru_res_preds
])

# Simple ensemble weighting
weights = np.array([0.35, 0.25, 0.25, 0.15])
master_forecast = np.dot(X_meta, weights)

print("🚀 Master Stacking Ensemble Forecast complete.")

## ✅ Step 4: Final Evaluation & Plotting

In [ ]:
def evaluate(actual, pred, name):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    print(f"--- {name} ---\nMAE : {mae:.2f}\nRMSE: {rmse:.2f}\n")

evaluate(test_actuals, arima_preds, "ARIMA Base")
evaluate(test_actuals, ets_preds, "ETS Base")
evaluate(test_actuals, master_forecast, "🏆 Master Stacking Ensemble")

plt.figure(figsize=(12, 6))
plt.plot(test_actuals.index, test_actuals, label='Actual', color='black', marker='o')
plt.plot(test_actuals.index, arima_preds, label='ARIMA Base', linestyle='--')
plt.plot(test_actuals.index, ets_preds, label='ETS Base', linestyle='--')
plt.plot(test_actuals.index, master_forecast, label='🏆 Master Ensemble', color='red', linewidth=3)
plt.title("The Master Stacking Ensemble vs. Individual Bases")
plt.legend()
plt.grid(True)
plt.show()